In [4]:
import pandas as pd
import numpy as np
import torch
import glob
import os
import warnings
from tqdm import tqdm
import logging
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'AppleGothic'  # macOS
warnings.filterwarnings('ignore')


# 루트 로거 재설정: 내 코드만 INFO, 외부 라이브러리는 WARNING 이상만
logging.basicConfig(level=logging.INFO, force=True)

# Matplotlib 쪽 디버그 차단
plt.set_loglevel("warning")
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("matplotlib.font_manager").setLevel(logging.WARNING)

# ensemble_model.py 파일 상단에 추가
import sys
sys.path.append('./')

# 기존 LSTM 모델 함수들 import
from baseline_lstm import (
    train_lstm, predict_lstm, generate_combined_holiday_list, 
    solar_md_holidays, lunar_solar_dates
)
# mene clipping import

# 클러스터링 모델 import  
from cluster_based_model import ClusterBasedForecastingModel

class EnsemblePredictor:
    """LSTM + 클러스터링 모델 앙상블"""
    
    def __init__(self, high_weight_venues=['담하', '미라시아']):
        self.high_weight_venues = high_weight_venues
        self.lstm_models = None
        self.cluster_model = None
        self.ensemble_weights = {}
        self.venue_performance = {}
        
    def calculate_smape(self, y_true, y_pred):
        """SMAPE 계산"""
        mask = y_true != 0
        if mask.sum() == 0:
            return 0
        y_true_filtered = y_true[mask]
        y_pred_filtered = y_pred[mask]
        return 100 * np.mean(2 * np.abs(y_pred_filtered - y_true_filtered) / 
                            (np.abs(y_true_filtered) + np.abs(y_pred_filtered)))
    
    def load_and_train_models(self, train_df_path='./train/train.csv'):
        """두 모델을 훈련합니다"""
        
        print("데이터 로딩 및 전처리...")
        train_df = pd.read_csv(train_df_path)
        
        # 매출 데이터 음수 제거
        train_df['매출수량'] = train_df['매출수량'].clip(lower=0)
        
        # 1. LSTM 모델 훈련
        print("LSTM 모델 훈련 중...")
        from korean_lunar_calendar import KoreanLunarCalendar
        
        # LSTM 전처리 (기존 코드와 동일)
        train_df = generate_combined_holiday_list(train_df, solar_md_holidays, lunar_solar_dates)
        self.lstm_models = train_lstm(train_df, use_validation=False, dropout=0.1)
        print(f"LSTM 모델 훈련 완료: {len(self.lstm_models)}개 모델")
        
        # 2. 클러스터링 모델 훈련
        print("클러스터링 모델 훈련 중...")
        cluster_train_df = train_df.copy()
        cluster_train_df['date'] = pd.to_datetime(cluster_train_df['영업일자'])
        cluster_train_df[['store', 'menu']] = cluster_train_df['영업장명_메뉴명'].str.split('_', expand=True, n=1)
        cluster_train_df['sales'] = cluster_train_df['매출수량']
        cluster_train_df['month'] = cluster_train_df['date'].dt.month
        cluster_train_df['day_of_week'] = cluster_train_df['date'].dt.dayofweek
        cluster_train_df['is_weekend'] = cluster_train_df['day_of_week'].isin([5, 6])
        
        self.cluster_model = ClusterBasedForecastingModel()
        self.cluster_model.fit(cluster_train_df)
        print("클러스터링 모델 훈련 완료")
    
    def _get_lstm_predictions(self, test_file):
        """LSTM 모델로 예측"""
        test_df = pd.read_csv(test_file)
        filename = os.path.basename(test_file)
        test_prefix = filename.replace('.csv', '')
        
        # LSTM 예측 수행 (기존 함수 활용)
        pred_df = predict_lstm(test_df, self.lstm_models, test_prefix)
        
        results = []
        for _, row in pred_df.iterrows():
            venue = row['영업장명_메뉴명']
            day = row['영업일자']
            value = row['매출수량']
            results.append({
                'venue': venue,
                'day': day, 
                'prediction': value
            })
        
        return results
    
    def _get_cluster_predictions(self, test_file):
        """클러스터링 모델로 예측"""
        test_df = pd.read_csv(test_file)
        
        # 클러스터링 모델 전처리
        cluster_test_df = test_df.copy()
        cluster_test_df['date'] = pd.to_datetime(cluster_test_df['영업일자'])
        cluster_test_df[['store', 'menu']] = cluster_test_df['영업장명_메뉴명'].str.split('_', expand=True, n=1)
        cluster_test_df['sales'] = cluster_test_df['매출수량']
        cluster_test_df['month'] = cluster_test_df['date'].dt.month
        cluster_test_df['day_of_week'] = cluster_test_df['date'].dt.dayofweek
        cluster_test_df['is_weekend'] = cluster_test_df['day_of_week'].isin([5, 6])
        
        # 클러스터링 예측
        predictions, metadata = self.cluster_model.predict(cluster_test_df)
        
        results = []
        filename = os.path.basename(test_file)
        test_prefix = filename.replace('.csv', '')
        
        for i, meta in enumerate(metadata):
            venue = f"{meta['store']}_{meta['menu']}"
            for day_idx in range(7):
                day = f"{test_prefix}+{day_idx+1}일"
                value = predictions[i][day_idx] if i < len(predictions) else 0
                results.append({
                    'venue': venue,
                    'day': day,
                    'prediction': value
                })
        
        return results
    
    def _match_predictions(self, lstm_results, cluster_results):
        """두 모델의 예측 결과 매칭"""
        
        # 딕셔너리로 변환하여 빠른 검색
        lstm_dict = {}
        for item in lstm_results:
            key = (item['venue'], item['day'])
            lstm_dict[key] = item['prediction']
        
        cluster_dict = {}
        for item in cluster_results:
            key = (item['venue'], item['day'])
            cluster_dict[key] = item['prediction']
        
        # 공통 키 찾기
        common_keys = set(lstm_dict.keys()) & set(cluster_dict.keys())
        
        # 업장별로 그룹화
        venue_groups = {}
        for venue, day in common_keys:
            if venue not in venue_groups:
                venue_groups[venue] = []
            venue_groups[venue].append((venue, day))
        
        matched_data = []
        for venue, keys in venue_groups.items():
            if len(keys) == 7:  # 7일치 모두 있을 때만
                keys_sorted = sorted(keys, key=lambda x: x[1])  # 날짜순 정렬
                
                lstm_pred = [lstm_dict[key] for key in keys_sorted]
                cluster_pred = [cluster_dict[key] for key in keys_sorted]
                target = [0] * 7  # 실제 타겟은 없으므로 0으로 설정 (검증용이 아니라면)
                
                matched_data.append({
                    'venue': venue,
                    'lstm_pred': lstm_pred,
                    'cluster_pred': cluster_pred,
                    'target': target
                })
        
        return matched_data
    
    def predict_ensemble(self, test_files):
        """앙상블 예측 수행"""
        
        if self.lstm_models is None or self.cluster_model is None:
            raise ValueError("모델이 훈련되지 않았습니다. load_and_train_models()를 먼저 실행하세요.")
        
        print("🎭 앙상블 예측 시작...")
        
        # 가중치가 설정되지 않았으면 기본값 사용
        if not hasattr(self, 'ensemble_weights') or not self.ensemble_weights:
            print("기본 가중치 사용 (LSTM 30%, 클러스터링 70%)")
            self.ensemble_weights = {'global': 0.3, 'venues': {}}
        
        all_predictions = []
        
        for test_file in tqdm(test_files, desc="테스트 파일 처리"):
            # 각 모델 예측
            lstm_results = self._get_lstm_predictions(test_file)
            cluster_results = self._get_cluster_predictions(test_file)
            
            # 예측 결과 매칭
            matched_data = self._match_predictions(lstm_results, cluster_results)
            
            # 앙상블 수행
            for item in matched_data:
                venue = item['venue']
                lstm_pred = np.array(item['lstm_pred'])
                cluster_pred = np.array(item['cluster_pred'])
                
                # 업장별 가중치 결정
                venue_name = venue.split('_')[0]
                if venue_name in self.ensemble_weights.get('venues', {}):
                    weight = self.ensemble_weights['venues'][venue_name]
                else:
                    weight = self.ensemble_weights['global']
                
                # 앙상블 예측
                ensemble_pred = weight * lstm_pred + (1 - weight) * cluster_pred
                
                # 후처리
                ensemble_pred = np.maximum(ensemble_pred, 1)  # 최소값 1
                
                # 결과 저장
                filename = os.path.basename(test_file)
                test_prefix = filename.replace('.csv', '')
                
                for day_idx, pred_value in enumerate(ensemble_pred):
                    all_predictions.append({
                        'test_file': test_prefix,
                        'venue': venue,
                        'day': f"{test_prefix}+{day_idx+1}일",
                        'lstm_pred': lstm_pred[day_idx],
                        'cluster_pred': cluster_pred[day_idx],
                        'ensemble_pred': pred_value,
                        'weight_used': weight
                    })
        
        return all_predictions
    
    def create_submission_file(self, predictions, sample_submission_path='./sample_submission.csv'):
        """제출 파일 생성"""
        
        sample_submission = pd.read_csv(sample_submission_path)
        submission = sample_submission.copy()
        
        # 예측 결과를 딕셔너리로 변환
        pred_dict = {}
        for pred in predictions:
            key = (pred['day'], pred['venue'])
            pred_dict[key] = pred['ensemble_pred']
        
        # 제출 파일에 매핑
        for row_idx in submission.index:
            date = submission.loc[row_idx, '영업일자']
            for col in submission.columns[1:]:  # 메뉴명들
                value = pred_dict.get((date, col), 0)
                submission.loc[row_idx, col] = max(0, value)
        
        return submission
    
        # =========================
    # 헬퍼: 검증 폴드 생성 (최근 28일 입력 + 다음 7일 정답)
    # =========================
    def _build_validation_fold(self, train_df, lookback=28, horizon=7):
        """
        각 '영업장명_메뉴명' 그룹에서 마지막 (lookback + horizon) 구간을 잘라
        검증 입력(마지막 28일)과 타깃(다음 7일)을 구성한다.
        반환:
          val_input_df : 예측용 입력 데이터프레임(28일 구간을 모두 이어붙임)
          y_true_map   : {(venue, 'VAL+{d}일'): y_true_value}
          w_map        : {(venue, 'VAL+{d}일'): sample_weight}  # 담하/미라시아 가중 2.0
        """
        val_rows = []
        y_true_map = {}
        w_map = {}

        for venue, g in train_df.groupby('영업장명_메뉴명'):
            g = g.sort_values('영업일자').copy()
            if len(g) < lookback + horizon:
                continue
            # 입력 28일
            X_win = g.iloc[-(lookback + horizon):-horizon].copy()
            X_win['영업장명_메뉴명'] = venue
            val_rows.append(X_win)
            # 타깃 7일
            y_win = g.iloc[-horizon:]['매출수량'].to_numpy()

            store = venue.split('_')[0]
            store_weight = 2.0 if store in self.high_weight_venues else 1.0
            for d in range(horizon):
                key = (venue, f"VAL+{d+1}일")
                y_true_map[key] = float(y_win[d])
                w_map[key] = float(store_weight)

        if not val_rows:
            return None, None, None

        val_input_df = pd.concat(val_rows, ignore_index=True)
        return val_input_df, y_true_map, w_map

    # =========================
    # 헬퍼: 검증 입력에 대해 두 모델 예측 수행 (test_prefix='VAL')
    # =========================
    def _predict_on_validation(self, val_input_df):
        """
        val_input_df: _build_validation_fold로 만든 28일 입력 모음
        반환:
          lstm_pred_map    : {(venue, 'VAL+{d}일'): yhat}
          cluster_pred_map : {(venue, 'VAL+{d}일'): yhat}
        """
        # 1) LSTM 예측 (기존 predict_lstm 사용)
        lstm_pred_map = {}
        lstm_pred_df = predict_lstm(val_input_df, self.lstm_models, test_prefix="VAL")  # (영업장명_메뉴명, 영업일자='VAL+1일', 매출수량=예측)
        for _, row in lstm_pred_df.iterrows():
            venue = row['영업장명_메뉴명']
            day   = row['영업일자']  # 'VAL+1일' ~ 'VAL+7일'
            val   = float(row['매출수량'])
            lstm_pred_map[(venue, day)] = val

        # 2) 클러스터 예측
        cluster_pred_map = {}
        cluster_df = val_input_df.copy()
        cluster_df['date'] = pd.to_datetime(cluster_df['영업일자'])
        cluster_df[['store', 'menu']] = cluster_df['영업장명_메뉴명'].str.split('_', expand=True, n=1)
        cluster_df['sales'] = cluster_df['매출수량']
        cluster_df['month'] = cluster_df['date'].dt.month
        cluster_df['day_of_week'] = cluster_df['date'].dt.dayofweek
        cluster_df['is_weekend'] = cluster_df['day_of_week'].isin([5, 6])

        preds, metadata = self.cluster_model.predict(cluster_df)
        # metadata[i] = {'store': ..., 'menu': ...}, preds[i] = 7-길이 시퀀스
        for i, meta in enumerate(metadata):
            venue = f"{meta['store']}_{meta['menu']}"
            seq   = preds[i] if i < len(preds) else [0]*7
            for d in range(7):
                day = f"VAL+{d+1}일"
                cluster_pred_map[(venue, day)] = float(seq[d])

        return lstm_pred_map, cluster_pred_map

    # =========================
    # 최적 가중치 탐색 (작동 버전)
    # =========================
    def find_optimal_weights_on_validation(self, train_df_path='./train/train.csv',
                                           lookback=28, horizon=7,
                                           grid_step=0.01, min_samples_for_special=21):
        """
        - train.csv의 최근 구간으로 검증 세트를 만들고(백테스트),
        - (담하/미라시아=2.0, 기타=1.0) 가중 SMAPE를 최소화하는
          전역 가중치와 업장별(담하/미라시아) 가중치를 찾는다.
        - 예측 후처리는 제출과 동일하게 최소 1로 클리핑.
        반환:
          {'global': w, 'venues': {'담하': w_damha, '미라시아': w_mirasia, ...}}
        """
        print("🔍 최적 가중치 탐색(백테스트) 시작...")

        # 0) 데이터 로드 & 전처리(학습때와 동일하게)
        train_df = pd.read_csv(train_df_path)
        train_df['매출수량'] = train_df['매출수량'].clip(lower=0)

        # LSTM 전처리와 동일한 휴일 특성 추가 (기존 함수를 그대로 사용)
        train_df = generate_combined_holiday_list(train_df, solar_md_holidays, lunar_solar_dates)

        # 1) 검증 폴드 생성
        val_input_df, y_true_map, w_map = self._build_validation_fold(train_df, lookback=lookback, horizon=horizon)
        if val_input_df is None:
            print("⚠️ 유효한 검증 폴드가 없습니다. 기본 가중치 사용 (global=0.3)")
            self.ensemble_weights = {'global': 0.3, 'venues': {}}
            return self.ensemble_weights

        # 2) 검증 입력에 대해 두 모델 예측
        lstm_pred_map, cluster_pred_map = self._predict_on_validation(val_input_df)

        # 3) 공통 키(venue, 'VAL+{d}일')에서 벡터 구성
        common_keys = set(y_true_map.keys()) & set(lstm_pred_map.keys()) & set(cluster_pred_map.keys())
        if not common_keys:
            print("⚠️ 예측/타깃 매칭 키가 없습니다. 기본 가중치 사용 (global=0.3)")
            self.ensemble_weights = {'global': 0.3, 'venues': {}}
            return self.ensemble_weights

        keys_sorted = sorted(common_keys, key=lambda x: (x[0], x[1]))
        y_true_vec     = np.array([y_true_map[k] for k in keys_sorted], dtype=float)
        lstm_vec       = np.array([lstm_pred_map[k] for k in keys_sorted], dtype=float)
        cluster_vec    = np.array([cluster_pred_map[k] for k in keys_sorted], dtype=float)
        weight_vec     = np.array([w_map[k] for k in keys_sorted], dtype=float)
        venue_name_vec = np.array([k[0].split('_')[0] for k in keys_sorted])

        # 4) 가중 SMAPE 함수
        def weighted_smape(y, yhat, w):
            mask = (y != 0)
            if mask.sum() == 0:
                return 0.0
            y2 = y[mask]; yhat2 = yhat[mask]; w2 = w[mask]
            sm = 2.0 * np.abs(yhat2 - y2) / (np.abs(y2) + np.abs(yhat2))
            return 100.0 * np.average(sm, weights=w2)

        # 5) 전역 가중치 탐색 (0~1, grid_step 간격)
        best_w, best_score = 0.5, float('inf')
        grid = np.arange(0.0, 1.0 + 1e-9, grid_step)
        for w in grid:
            yhat = w * lstm_vec + (1.0 - w) * cluster_vec
            yhat = np.maximum(yhat, 1.0)  # 제출과 동일한 하한
            sc = weighted_smape(y_true_vec, yhat, weight_vec)
            if sc < best_score:
                best_score, best_w = sc, w

        # 6) 특별 업장(담하/미라시아 등) 개별 최적화
        venue_weights = {}
        for v in self.high_weight_venues:
            mask = (venue_name_vec == v)
            if mask.sum() >= min_samples_for_special:
                y_true_v  = y_true_vec[mask]
                lstm_v    = lstm_vec[mask]
                cluster_v = cluster_vec[mask]
                w_v       = weight_vec[mask]

                best_v_w, best_v_sc = best_w, float('inf')
                for w in grid:
                    yhat_v = w * lstm_v + (1.0 - w) * cluster_v
                    yhat_v = np.maximum(yhat_v, 1.0)
                    sc_v = weighted_smape(y_true_v, yhat_v, w_v)
                    if sc_v < best_v_sc:
                        best_v_sc, best_v_w = sc_v, w
                venue_weights[v] = float(best_v_w)

        self.ensemble_weights = {'global': float(best_w), 'venues': venue_weights}
        print(f"✅ 최적 가중치 - 전역: {best_w:.2f}, 특별업장: {venue_weights} (전역 점수 {best_score:.4f})")
        return self.ensemble_weights


# 사용 예시
def run_ensemble_pipeline():
    """전체 앙상블 파이프라인 실행"""
    
    print("🚀 앙상블 파이프라인 시작!")
    
    # 1. 앙상블 모델 초기화
    ensemble = EnsemblePredictor(high_weight_venues=['담하', '미라시아'])
    
    # 2. 두 모델 훈련
    ensemble.load_and_train_models('./train/train.csv')
    
    # 3. 최적 가중치 설정
    ensemble.find_optimal_weights_on_validation(train_df_path='./train/train.csv')
    
    # 4. 테스트 예측
    test_files = sorted(glob.glob('./TEST_*.csv'))
    predictions = ensemble.predict_ensemble(test_files)
    
    # 5. 제출 파일 생성
    submission = ensemble.create_submission_file(predictions)
    
    # 6. 저장
    output_path = './ens_sub_4.csv'
    submission.to_csv(output_path, index=False, encoding='utf-8-sig')
    
    print(f"✅ 앙상블 완료! 저장 위치: {output_path}")
    
    # 7. 결과 분석
    print("\n📊 앙상블 결과 분석:")
    pred_df = pd.DataFrame(predictions)
    
    print(f"총 예측 수: {len(pred_df)}")
    print(f"평균 LSTM 예측: {pred_df['lstm_pred'].mean():.2f}")
    print(f"평균 클러스터링 예측: {pred_df['cluster_pred'].mean():.2f}")
    print(f"평균 앙상블 예측: {pred_df['ensemble_pred'].mean():.2f}")
    
    weight_stats = pred_df['weight_used'].value_counts()
    print(f"사용된 가중치 분포:\n{weight_stats}")
    
    return submission, pred_df

# ===== 실행 방법 =====
"""
1. 이 코드를 ensemble_model.py로 저장
2. 기존 모델 코드들이 같은 디렉토리에 있는지 확인
3. 다음 명령 실행:

from ensemble_model import run_ensemble_pipeline
submission, predictions = run_ensemble_pipeline()

또는 단계별로:

ensemble = EnsemblePredictor()
ensemble.load_and_train_models()
test_files = glob.glob('./test/TEST_*.csv')
predictions = ensemble.predict_ensemble(test_files)
submission = ensemble.create_submission_file(predictions)
submission.to_csv('ensemble_result.csv', index=False)
"""

# if __name__ == "__main__":
#     submission, predictions = run_ensemble_pipeline()

"\n1. 이 코드를 ensemble_model.py로 저장\n2. 기존 모델 코드들이 같은 디렉토리에 있는지 확인\n3. 다음 명령 실행:\n\nfrom ensemble_model import run_ensemble_pipeline\nsubmission, predictions = run_ensemble_pipeline()\n\n또는 단계별로:\n\nensemble = EnsemblePredictor()\nensemble.load_and_train_models()\ntest_files = glob.glob('./test/TEST_*.csv')\npredictions = ensemble.predict_ensemble(test_files)\nsubmission = ensemble.create_submission_file(predictions)\nsubmission.to_csv('ensemble_result.csv', index=False)\n"

In [6]:
from clipping import SimpleEffectiveClipping
clipper = SimpleEffectiveClipping('./train.csv')
df = pd.read_csv('./ens/ens_75_25.csv')
clipped_df = clipper.apply_method_3_p90(df)

clipped_df.to_csv('./ens/ens_75_25_clipped.csv',index=False, encoding="utf-8-sig")

클리핑 제외 업장: []
클리핑 제외 메뉴 수: 0
클리핑 적용 메뉴 수: 193
  → 클리핑 적용: 193개, 제외: 0개 메뉴


In [4]:
submission, predictions = run_ensemble_pipeline()

🚀 앙상블 파이프라인 시작!
데이터 로딩 및 전처리...
LSTM 모델 훈련 중...


Training LSTM: 100%|██████████| 193/193 [22:28<00:00,  6.99s/it]


LSTM 모델 훈련 완료: 193개 모델
클러스터링 모델 훈련 중...


클러스터 모델:  17%|█▋        | 1/6 [00:02<00:11,  2.24s/it]

클러스터 0 완료: group, 5251개 샘플


...

클러스터 5 완료: other, 46626개 샘플
클러스터링 모델 훈련 완료
🔍 최적 가중치 탐색(백테스트) 시작...


예측 진행: 100%|██████████| 193/193 [00:00<00:00, 300.13it/s]


✅ 최적 가중치 - 전역: 0.18, 특별업장: {'담하': 0.18, '미라시아': 0.66} (전역 점수 47.4089)
🎭 앙상블 예측 시작...


테스트 파일 처리: 100%|██████████| 10/10 [00:18<00:00,  1.83s/it]


✅ 앙상블 완료! 저장 위치: ./ens_sub_4.csv

📊 앙상블 결과 분석:
총 예측 수: 13510
평균 LSTM 예측: 7.77
평균 클러스터링 예측: 9.43
평균 앙상블 예측: 9.28
사용된 가중치 분포:
weight_used
0.18    11340
0.66     2170
Name: count, dtype: int64
